In [2]:
import pandas as pd
from thefuzz import process

import sys
import os

# Go up one level from 'notebooks' to the project root
sys.path.append(os.path.abspath(".."))

In [3]:
df = pd.read_excel("../data/google_contacts.xlsx")
df.head()

,First Name,Last Name,Email,Phone,Address
0,ERIKA,Richmond,gilbert48_at_example.org,+49 682777013,Russell Center 100
1,CRAIG,Vasquez,lori16_at_example.org,969747690,Cox Forest Street 2
2,Philip,Hess,colleenrodriguez@example.net,917061765,Timothy Station 99
3,Emily,Meyers,browndaniel@example.org,0049-293636128,Johnson Plaza 70
4,Justin,Sharp,johnsonjoe@example.com,007768876,Mark Mountain Street 21


In [4]:
df.columns

Index(['First Name', 'Last Name', 'Email', 'Phone', 'Address'], dtype='str')

In [5]:
xero_df = pd.read_csv("../data/xero_contacts.csv")
xero_df.head()

,Full Name,Email Address,Phone Number,Company,Address Line 1
0,Taylor Davis,shaneleonard@example.net,0049-989085852,Cole Group,Derek Prairie 29
1,Andrew Smith,suarezmichael@example.org,+49-681-864470,"Smith, Perez and Lyons",5 Perez Passage
2,Jose Brown,oliviapeters@example.org,135357404,Espinoza Ltd,Payne Fords 1
3,Stephanie Weaver,burnettkimberly@example.com,+49-705-792701,"Contreras, Ferguson and David",44 Amanda Roads St
4,Cindy Webb,samanthavargas@example.org,+49 029535403,"Schaefer, Conway and Anderson",49 Meadows Overpass


In [6]:
xero_df.columns

Index(['Full Name', 'Email Address', 'Phone Number', 'Company',
       'Address Line 1'],
      dtype='str')

Customer list show data integration or schema mapping problem:

- Phone numbers, emails are not consistent / uniform
- Google file has different colunm names / headers from those in xero file e.g., address vs address line 1
- While xero file has full names, google has first and last names. 

## Cleaning 

In [7]:
#import the module from my deep_clean library
from deep_clean import standardize_headers

# example of what standardize_headers does
xero_df = standardize_headers(xero_df)
google_df = standardize_headers(df)

print(xero_df.columns)
print('\n')
print(google_df.columns)

Index(['full_name', 'email', 'phone', 'company', 'address'], dtype='str')


Index(['first_name', 'last_name', 'email', 'phone', 'address'], dtype='str')


In [8]:
xero_df.head()

,full_name,email,phone,company,address
0,Taylor Davis,shaneleonard@example.net,0049-989085852,Cole Group,Derek Prairie 29
1,Andrew Smith,suarezmichael@example.org,+49-681-864470,"Smith, Perez and Lyons",5 Perez Passage
2,Jose Brown,oliviapeters@example.org,135357404,Espinoza Ltd,Payne Fords 1
3,Stephanie Weaver,burnettkimberly@example.com,+49-705-792701,"Contreras, Ferguson and David",44 Amanda Roads St
4,Cindy Webb,samanthavargas@example.org,+49 029535403,"Schaefer, Conway and Anderson",49 Meadows Overpass


In [9]:
# ensure every customer has a full name
# when full_name exists, it skips, otherwise combine first and last to create full_name
from deep_clean import ensure_full_name

xero_df = ensure_full_name(xero_df)
google_df = ensure_full_name(google_df)

print(xero_df.head(3))
print('\n')
print(google_df.head(3))

      full_name                      email           phone  \
0  Taylor Davis   shaneleonard@example.net  0049-989085852   
1  Andrew Smith  suarezmichael@example.org  +49-681-864470   
2    Jose Brown   oliviapeters@example.org       135357404   

                  company           address  
0              Cole Group  Derek Prairie 29  
1  Smith, Perez and Lyons   5 Perez Passage  
2            Espinoza Ltd     Payne Fords 1  


                          email          phone              address  \
0      gilbert48_at_example.org  +49 682777013   Russell Center 100   
1         lori16_at_example.org      969747690  Cox Forest Street 2   
2  colleenrodriguez@example.net      917061765   Timothy Station 99   

        full_name  
0  ERIKA Richmond  
1   CRAIG Vasquez  
2     Philip Hess  


In [10]:
# combine
combined_df = pd.concat([google_df, xero_df], ignore_index=True)

# Fill NaN in Company with an Not available
combined_df['company'] = combined_df['company'].fillna('Individual')

combined_df.head()

,email,phone,address,full_name,company
0,gilbert48_at_example.org,+49 682777013,Russell Center 100,ERIKA Richmond,Individual
1,lori16_at_example.org,969747690,Cox Forest Street 2,CRAIG Vasquez,Individual
2,colleenrodriguez@example.net,917061765,Timothy Station 99,Philip Hess,Individual
3,browndaniel@example.org,0049-293636128,Johnson Plaza 70,Emily Meyers,Individual
4,johnsonjoe@example.com,007768876,Mark Mountain Street 21,Justin Sharp,Individual


In [11]:
# clean address
from deep_clean import clean_addresses
combined_df = clean_addresses(combined_df)
combined_df.head()

,email,phone,address,full_name,company
0,gilbert48_at_example.org,+49 682777013,Russell Center 100,ERIKA Richmond,Individual
1,lori16_at_example.org,969747690,Cox Forest Street 2,CRAIG Vasquez,Individual
2,colleenrodriguez@example.net,917061765,Timothy Station 99,Philip Hess,Individual
3,browndaniel@example.org,0049-293636128,Johnson Plaza 70,Emily Meyers,Individual
4,johnsonjoe@example.com,007768876,Mark Mountain Street 21,Justin Sharp,Individual


## Load and inspect clean customer data after running the pipeline

In [12]:
# Load and inspect clean customer data after running the pipeline
df = pd.read_csv("../data/clean_customers.csv")
df.head()

,email,phone,address,full_name,company,is_valid_email
0,gilbert48@example.org,49682777013,Russell Center 100,Erika Richmond,Individual,True
1,lori16@example.org,969747690,Cox Forest Street 2,Craig Vasquez,Individual,True
2,colleenrodriguez@example.net,917061765,Timothy Station 99,Philip Hess,Individual,True
3,browndaniel@example.org,49293636128,Johnson Plaza 70,Emily Meyers,Individual,True
4,johnsonjoe@example.com,7768876,Mark Mountain Street 21,Justin Sharp,Individual,True


In [13]:
df.tail(10)

,email,phone,address,full_name,company,is_valid_email
141,xanderson@example.com,49448417676,99 Alexander Place St,Caitlin Hughes,Walton-Moore,True
142,agood@example.com,593994042,Cannon Cove Street 83,Oscar Walker,Gibbs-Maxwell,True
143,angelahenry@example.com,49198790664,Cynthia Trafficway Street 14,Kimberly Atkinson,"Barnes, Smith and Ramirez",True
144,cmullins@example.net,49841504857,93 Kristin Ports St,Erik Bell,French-Grant,True
145,elizabethdixon@example.com,831369556,Warren Port Street 86,Helen Richardson,Moore-Lutz,True
146,lscott@example.org,49340651456,Abigail Mills Street 25,Scott Hester,Jackson-Rios,True
147,fjackson@example.net,49816441206,Walker Islands 87,Anthony Bonilla,Taylor-Barber,True
148,robertstimothy@example.org,274339329,52 Turner Greens,Pamela Edwards,Daniels-Leach,True
149,tracynorman@example.com,348873943,Andrews Terrace 37,Benjamin Harrison,"Robles, Oconnor and Fox",True
150,millerjoshua@example.org,49304959190,75 Kimberly Road St,Jessica Cain,Richardson-Mills,True


- Now all customers have full name, email, phone, address and company. The columns were named slightly different in the two datasets but they are now unified. 

- All emails have been properly formatted to e.g., xanderson@example.com

- The phone numbers are also properly formatted to only digits. No underscore, dashes, or plus signs

In [14]:
failed = df[~df['is_valid_email']]
failed[["full_name", "email"]]

,full_name,email
13,Kimberly Bryant,no_email_provided


- Kimberly Bryant did not provide his/her email address and that's why in the pipeline, one email failed validation because it doesn't exist.

- Depending on the business wants to use this customer list for, they may decide to:
    1. Drop customers without emails especially when relying only on email marketing or
    2. Keep them as they are valid customers and label the email missing in cases where email is not needed.